<a href="https://colab.research.google.com/github/rselvak6/opendde-colab/blob/main/opendde_monomer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenDDE on Colab

Requires a GPU runtime: **suggested: A100 high-ram**

This notebook: installs OpenDDE from source with `uv`, downloads model weights + inference data, runs a tiny smoke test, then runs your real prediction job.

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!test -d OpenDDE || git clone https://github.com/aurekaresearch/OpenDDE.git
%cd OpenDDE
!uv pip install --torch-backend cu126 -e '.[gpu]'
!uv pip install --group dev
!opendde doctor

downloading uv 0.11.31 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Cloning into 'OpenDDE'...
remote: Enumerating objects: 586, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 586 (delta 15), reused 12 (delta 11), pack-reused 559 (from 1)
Receiving objects: 100% (586/586), 20.84 MiB | 20.21 MiB/s, done.
Resolving deltas: 100% (266/266), done.
/content/OpenDDE
Using Python 3.12.13 environment at: /usr
Resolved 65 packages in 1.53s
Prepared 31 packages in 46.80s
Uninstalled 21 packages in 749ms
Installed 31 packages in 196ms
 + biopython==1.85
 + biotite==1.4.0
 + biotraj==1.2.2
 + cuequivariance==0.10.0
 + cuequivariance-ops-cu12==0.10.0
 + cuequivariance-ops-torch-cu12==0.10.0
 + cuequivariance-torch==0.10.0
 + ml-collections==1.1.0
 - numpy==2.0.2
 + numpy==2.4.1
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.6.4.1
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-

In [ ]:
# General-purpose checkpoint used by default with -n opendde_v1.
%env OPENDDE_ROOT_DIR=/content/data
!mkdir -p $OPENDDE_ROOT_DIR/checkpoint

env: OPENDDE_ROOT_DIR=/content/data


In [ ]:
# Download only what this notebook actually uses.
!apt-get -qq update && apt-get -qq install -y zstd
!bash /content/OpenDDE/scripts/download_opendde_data.sh --skip-search-database

In [ ]:
import os
os.environ["OPENDDE_FORCE_SAMPLE_DIFFUSION_AMP"] = "1"
os.environ["OPENDDE_FORCE_CONFIDENCE_AMP"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# Must match the "name" field of the input JSON: OpenDDE writes results to
# ./output/<name>/, which every downstream cell reads, and the run cell below
# loads /content/OpenDDE/<PROJECT_NAME>.json. Pick a known input or type one.
PROJECT_NAME = 'egfr-rad51-monomer'  # @param ["egfr-rad51-monomer", "egfr-rad51-monomer-afatinib", "egfr-rad51-monomer-alflutinib", "egfr-rad51-monomer-bh30643", "egfr-rad51-monomer-erlotinib", "egfr-rad51-monomer-gefitinib", "egfr-rad51-monomer-lazertinib", "egfr-rad51-monomer-osimertinib"] {allow-input: true}

In [ ]:
TEMPLATE = 'none'  # @param ["none", "ideal-inactive", "ideal-active"] {allow-input: true}

In [ ]:
!opendde pred -i /content/OpenDDE/{PROJECT_NAME}.json -o ./output -n opendde_v1 --use_msa true --use_template false --use_rna_msa false --sample 1 --step 200 --cycle 10 --need_atom_confidence true --dtype bf16

2026-07-22 15:10:31,629 [/usr/local/lib/python3.12/dist-packages/numexpr/utils.py:164] INFO numexpr.utils: NumExpr defaulting to 12 threads.
2026-07-22 15:10:32,611 [/content/OpenDDE/runner/batch_inference.py:857] INFO runner.batch_inference: Run infer with input=/content/OpenDDE/egfr-rad51-fusion-cetuximab.json, out_dir=./output, sample=1
2026-07-22 15:10:32,611 [/content/OpenDDE/runner/batch_inference.py:866] INFO runner.batch_inference: Using inference params for model opendde_v1: cycle=10, step=200, use_msa=True
2026-07-22 15:10:32,611 [/content/OpenDDE/runner/batch_inference.py:488] INFO runner.batch_inference: Will infer with 1 jsons
2026-07-22 15:10:32,626 [/content/OpenDDE/runner/inference.py:361] INFO runner.inference: Distributed environment: world size: 1, global rank: 0, local rank: 0
2026-07-22 15:10:32,777 [/content/OpenDDE/runner/inference.py:170] INFO root: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
2026-07-22 15:10:35,175 [/content/OpenDDE/opendde/config/inference.py:11

## Post-processing: visualize structures + rank by ipSAE

Requires the `pred` runs above to have used `--need_atom_confidence true` (adds `*_full_data_sample_*.json` per model, with the PAE matrix and per-atom pLDDT).

In [ ]:
# @title Shared helpers: locate each model's .cif plus its matching per-sample JSON
# sidecars, following OpenDDE's dumper.py naming convention:
#   {name}_sample_{rank}.cif
#   {name}_summary_confidence_sample_{rank}.json   (always written)
#   {name}_full_data_sample_{rank}.json             (only with --need_atom_confidence true)
import glob
import os
import re

_SAMPLE_RE = re.compile(r"(.+)_sample_(\d+)\.cif$")


def find_all_cifs(output_dir="./output/{}".format(PROJECT_NAME)):
    return sorted(glob.glob(os.path.join(output_dir, "**", "*.cif"), recursive=True))


def _sample_stem_rank(cif_path):
    m = _SAMPLE_RE.match(os.path.basename(cif_path))
    return m.groups() if m else (None, None)


def find_full_data_json(cif_path):
    stem, rank = _sample_stem_rank(cif_path)
    if stem is None:
        return None
    p = os.path.join(os.path.dirname(cif_path), f"{stem}_full_data_sample_{rank}.json")
    return p if os.path.exists(p) else None


def find_summary_json(cif_path):
    stem, rank = _sample_stem_rank(cif_path)
    if stem is None:
        return None
    p = os.path.join(os.path.dirname(cif_path), f"{stem}_summary_confidence_sample_{rank}.json")
    return p if os.path.exists(p) else None


def model_label(cif_path):
    # Short "seed N, sample M" label instead of the full relative path.
    _, rank = _sample_stem_rank(cif_path)
    seed_match = re.search(r"seed_(\d+)", cif_path)
    seed = seed_match.group(1) if seed_match else "?"
    return f"seed {seed}, sample {rank}" if rank is not None else os.path.basename(cif_path)


cifs = find_all_cifs()
print(f"Found {len(cifs)} predicted structure(s) under ./output/{PROJECT_NAME}")

Found 0 predicted structure(s) under ./output/egfr-full


In [ ]:
#@title Visualize predicted structures + PAE (py2Dmol)
!pip install -q py3Dmol py2Dmol

import json

import numpy as np
import py2Dmol

load_as_frames = True  # @param {type:"boolean"}
viewer_size = 400


def load_pae(cif_path):
    # OpenDDE's full_data.json stores the PAE matrix as "token_pair_pae"
    # (AlphaFold3's equivalent full_data.json key is "pae").
    fd_path = find_full_data_json(cif_path)
    if fd_path is None:
        return None
    with open(fd_path) as f:
        data = json.load(f)
    pae = data.get("token_pair_pae")
    return np.asarray(pae, dtype=float) if pae is not None else None


if not cifs:
    print("No .cif files found under ./output yet - run a prediction first.")
else:
    viewer = py2Dmol.view(size=(viewer_size, viewer_size), pae=True, autoplay=load_as_frames)
    for i, cif in enumerate(cifs, start=1):
        pae = load_pae(cif)
        if pae is None:
            print(f"No PAE data for {cif} (re-run pred with --need_atom_confidence true)")
        name = "models" if load_as_frames else os.path.basename(cif).replace(".cif", "")
        viewer.add_pdb(cif, name=name, paes=pae)
    viewer.show()

No .cif files found under ./output yet - run a prediction first.


## Utilities

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("{}".format(PROJECT_NAME), "zip", "./output/{}".format(PROJECT_NAME))
files.download("{}.zip".format(PROJECT_NAME))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import glob, os

for pattern in ("*.txt", "*.pml"):
    for f in glob.glob(f"./output/**/{pattern}", recursive=True):
        os.remove(f)